# Credit Card Fraud Detection — 7-Step Analytics Pipeline

This notebook demonstrates an end-to-end, interview-ready credit card fraud detection pipeline designed for **Synchrony Analytics** roles.

### Pipeline Overview:
1. **Load Data & Basic Understanding** (Severe class imbalance: 0.17% fraud)
2. **Visualize Class Imbalance** (Why naive accuracy fails)
3. **Train/Test Split & Robust Scaling** (Preventing data leakage)
4. **SMOTE Oversampling** (Applied strictly to training data)
5. **Interpretable Logistic Regression** (Probabilistic baseline)
6. **Evaluation Metrics** (Precision, Recall, F1, PR-AUC, Confusion Matrix)
7. **Business-Based Threshold Optimization** (Minimizing estimated financial cost)


## Cell 1 — Load Data & Basic Understanding

Load the dataset using `kagglehub`.

> **The dataset contains 284,807 transactions and only 492 fraud cases, creating severe class imbalance.**

In [1]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))

print(df.shape)
print(df["Class"].value_counts())
print(df.isnull().sum().sum())

Dataset Shape: (284807, 31)

Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Total Missing Values: 0


## Cell 2 — Visualize Class Imbalance

Because fraud represents only ~0.17% of transactions, a naive model predicting "Legitimate" 100% of the time achieves **99.83% accuracy** while catching zero fraud. Accuracy alone is misleading.

In [2]:
import matplotlib.pyplot as plt

counts = df["Class"].value_counts()
total = len(df)
fraud_pct = (counts[1] / total) * 100
legit_pct = (counts[0] / total) * 100

print(f"Legitimate: {counts[0]:,} ({legit_pct:.3f}%)")
print(f"Fraudulent: {counts[1]:,} ({fraud_pct:.3f}%)")

plt.figure(figsize=(6, 4))
plt.bar(["Legitimate", "Fraudulent"], [counts[0], counts[1]], color=["#2b5c8f", "#d62728"], width=0.4)
plt.yscale("log")
plt.title("Class Distribution (Log Scale)")
plt.ylabel("Transaction Count")
plt.tight_layout()
plt.show()

Legitimate: 284,315 (99.827%)
Fraudulent: 492 (0.173%)


## Cell 3 — Train/Test Split + Robust Scaling

### Data Leakage Note:
We fit `RobustScaler` **ONLY on `X_train`** and then transform `X_test`. Fitting a scaler across the entire dataset leaks unseen test-set distribution statistics (medians/IQRs) into the training process, invalidating realistic performance estimates.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (227845, 30), X_test shape: (56962, 30)


## Cell 4 — SMOTE (Synthetic Minority Over-sampling Technique)

SMOTE generates synthetic minority instances to bring the fraud class to a 20% ratio (`sampling_strategy=0.2`).

**Crucial Rule**: SMOTE must **only** be applied to training data. The test set (`X_test`, `y_test`) must remain untouched to evaluate real-world performance accurately.

In [4]:
from imblearn.over_sampling import SMOTE

print("Class distribution before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(
    sampling_strategy=0.2,
    random_state=42
)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

print("\nClass distribution after SMOTE:")
print(y_train.value_counts())

Class distribution before SMOTE:
Class
0    227451
1       394
Name: count, dtype: int64

Class distribution after SMOTE:
Class
0    227451
1     45490
Name: count, dtype: int64


## Cell 5 — Logistic Regression

We train an interpretable Logistic Regression model. The goal is clear, explainable analytics reasoning and calibrated probability scoring.

In [5]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Logistic Regression fitted successfully.")
print(f"Feature weights shape: {model.coef_.shape}")

Logistic Regression fitted successfully.
Feature weights shape: (1, 30)


## Cell 6 — Model Evaluation

> **Because fraud represents only about 0.17% of transactions, accuracy alone can be misleading. Precision, recall and PR-AUC provide a better view of minority-class performance.**

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

cm = confusion_matrix(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_prob)

print("Confusion Matrix (Default 0.50 Threshold):\n", cm)
print(f"\nPR-AUC Score: {pr_auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

Confusion Matrix (Default 0.50 Threshold):
[[56527   337]
 [   11    87]]

PR-AUC Score: 0.7388

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9941    0.9969     56864
           1     0.2052    0.8878    0.3333        98

    accuracy                         0.9939     56962
   macro avg     0.6025    0.9409    0.6651     56962
weighted avg     0.9984    0.9939    0.9958     56962



## Cell 7 — Business-Based Threshold Optimization

Instead of blindly assuming a 0.50 threshold, we optimize the decision threshold based on business costs.

> **Illustrative Business Cost Assumptions (not actual Synchrony costs):**
- **False Negative (missed fraud):** \$150
- **False Positive (unnecessary fraud alert / customer friction):** \$5

$$\text{Financial Cost} = (\text{FN} \times 150) + (\text{FP} \times 5)$$

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Illustrative business costs (not actual Synchrony values)
COST_FN = 150  # Missed fraud cost
COST_FP = 5    # False alarm friction cost

thresholds = np.linspace(0.10, 0.90, 81)
costs = []
records = []

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    cost = fn * COST_FN + fp * COST_FP
    costs.append(cost)
    records.append((t, cost, tn, fp, fn, tp))

min_idx = np.argmin(costs)
best_t, min_cost, b_tn, b_fp, b_fn, b_tp = records[min_idx]

def_preds = (y_prob >= 0.50).astype(int)
d_tn, d_fp, d_fn, d_tp = confusion_matrix(y_test, def_preds).ravel()
default_cost = d_fn * COST_FN + d_fp * COST_FP

print(f"Best Threshold: {best_t:.2f}")
print(f"Minimum Estimated Cost: ${min_cost:,} (TN: {b_tn:,}, FP: {b_fp:,}, FN: {b_fn:,}, TP: {b_tp:,})")
print(f"Default Threshold 0.50 Cost: ${default_cost:,} (TN: {d_tn:,}, FP: {d_fp:,}, FN: {d_fn:,}, TP: {d_tp:,})")
print(f"Net Dollar Savings: ${default_cost - min_cost:,} (-{((default_cost - min_cost) / default_cost) * 100:.1f}% reduction)")

# Plot Decision Threshold vs Estimated Financial Cost
plt.figure(figsize=(8, 5))
plt.plot(thresholds, costs, color="#2b5c8f", linewidth=2.5, label="Estimated Cost ($)")
plt.axvline(best_t, color="#d62728", linestyle="--", linewidth=1.5,
            label=f"Optimal Threshold = {best_t:.2f} (${min_cost:,})")
plt.axvline(0.50, color="#7f7f7f", linestyle=":", linewidth=1.5,
            label=f"Default Threshold = 0.50 (${default_cost:,})")
plt.scatter([best_t], [min_cost], color="#d62728", s=80, zorder=5)
plt.title("Decision Threshold vs Estimated Financial Cost\n(FN=\$150, FP=\$5 Illustrative Business Costs)")
plt.xlabel("Decision Threshold")
plt.ylabel("Estimated Financial Cost ($)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

Best Threshold: 0.84
Minimum Estimated Cost: $2,045 (TN: 56,785, FP: 79, FN: 11, TP: 87)
Default Threshold 0.50 Cost: $3,335 (TN: 56,527, FP: 337, FN: 11, TP: 87)
Net Dollar Savings: $1,290 (-38.7% reduction)
